# 03 - Representation B: imputation + missingness masks
**Run in Google Colab, after notebook 02.** Needs these files in your Drive folder: `sepsis_merged.csv`, `repA_train.csv`, `repA_val.csv`.

**Design (a controlled comparison):** B = exactly the same rows and the same imputed features as A, plus one extra
0/1 column per lab/vital: `<col>_measured` = 1 if that value was actually recorded at that hour (before any filling).
Same split, same models, same settings. The only thing that changes is the mask columns.
The test split is deliberately not loaded here.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS: same Drive folder you used in notebook 02
DRIVE_DIR = '/content/drive/MyDrive/sepsis'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import datetime
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, f1_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.utils.class_weight import compute_sample_weight

VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos",
        "Calcium", "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate",
        "Magnesium", "Phosphate", "Potassium", "Bilirubin_total", "TroponinI", "Hct",
        "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"]
MASK_COLS = VITALS + LABS                      # 34 columns, as in the official data description
MASK_FEATURES = [f"{c}_measured" for c in MASK_COLS]
SEEDS = [42, 43, 44]                           # HistGradientBoosting is run once per seed

## 1. Load Representation A (train + val only)
These files come from notebook 02, so the patient split is identical by construction.

In [5]:
A_train = pd.read_csv(f"{DRIVE_DIR}/repA_train.csv")
A_val = pd.read_csv(f"{DRIVE_DIR}/repA_val.csv")
print(f"A_train: {A_train.shape}, A_val: {A_val.shape}")

A_train: (1086436, 45), A_val: (231472, 45)


## 2. Build the missingness masks from the RAW data
The masks must come from the raw merged file: after notebook 02 the gaps are already filled, so the information
about what was really measured can only be recovered from `sepsis_merged.csv`. We join them onto the A rows
by (`patient_id`, `hour`), so every other feature stays exactly as it was in A.

In [6]:
raw = pd.read_csv(f"{DRIVE_DIR}/sepsis_merged.csv", usecols=["patient_id", "hour"] + MASK_COLS)
masks = raw[["patient_id", "hour"]].copy()
for c in MASK_COLS:
    masks[f"{c}_measured"] = raw[c].notna().astype("int8")
del raw

def add_masks(a_df):
    out = a_df.merge(masks, on=["patient_id", "hour"], how="left", validate="one_to_one")
    assert len(out) == len(a_df), "row count changed during merge"
    assert out[MASK_FEATURES].notna().all().all(), "some rows have no mask values"
    return out

B_train = add_masks(A_train)
B_val = add_masks(A_val)

print("Share of hours each feature is actually measured (rarest 6):")
print((B_train[MASK_FEATURES].mean() * 100).sort_values().head(6).round(2).to_string())

Share of hours each feature is actually measured (rarest 6):
Bilirubin_direct_measured    0.19
Fibrinogen_measured          0.67
TroponinI_measured           0.96
Bilirubin_total_measured     1.49
Alkalinephos_measured        1.61
AST_measured                 1.63


## 3. How much does "was it measured?" tell us about sepsis, on its own?
Mutual information (in nats) between each mask and the label, computed on the training split. This needs no model.
**Caveat for the report:** masks are also correlated with time in the ICU (more tests happen later in a stay, and positive
labels also appear later), so a high value here is a hint, not proof. The model comparison below is the real test.

In [7]:
mi = mutual_info_classif(B_train[MASK_FEATURES], B_train["SepsisLabel"],
                         discrete_features=True, random_state=0)
mi_series = pd.Series(mi, index=MASK_FEATURES).sort_values(ascending=False)
print("Top 10 masks by mutual information with SepsisLabel:")
print(mi_series.head(10).map(lambda v: f"{v:.5f}").to_string())

Top 10 masks by mutual information with SepsisLabel:
FiO2_measured            0.00086
EtCO2_measured           0.00083
Lactate_measured         0.00047
pH_measured              0.00047
PaCO2_measured           0.00045
BaseExcess_measured      0.00033
SaO2_measured            0.00022
Potassium_measured       0.00015
Alkalinephos_measured    0.00012
AST_measured             0.00012


## 4. Train identical models on A and on B
Logistic Regression is deterministic, so it runs once. HistGradientBoosting holds out a random slice of the training
rows for early stopping, so its result changes a little with the seed: we run 3 seeds and report mean and std,
so that we can tell a real gain from noise. Same class-balanced sample weights and threshold 0.5 as notebook 02.

In [8]:
BASE_FEATURES = [c for c in A_train.columns if c not in ("patient_id", "SepsisLabel")]
FEATURE_SETS = {"A": BASE_FEATURES, "B": BASE_FEATURES + MASK_FEATURES}
print({k: len(v) for k, v in FEATURE_SETS.items()}, "features")

y_train, y_val = B_train["SepsisLabel"], B_val["SepsisLabel"]
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

def score(probs):
    return dict(pr_auc=average_precision_score(y_val, probs),
                f1=f1_score(y_val, (probs >= 0.5).astype(int)))

results = []
for rep, cols in FEATURE_SETS.items():
    X_tr, X_va = B_train[cols], B_val[cols]

    scaler = StandardScaler().fit(X_tr)
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(scaler.transform(X_tr), y_train, sample_weight=sample_weight)
    results.append(dict(rep=rep, model="LogisticRegression", seed=42,
                        **score(logreg.predict_proba(scaler.transform(X_va))[:, 1])))
    print(f"[{rep}] LogisticRegression done")

    for seed in SEEDS:
        hgb = HistGradientBoostingClassifier(random_state=seed)
        hgb.fit(X_tr, y_train, sample_weight=sample_weight)
        results.append(dict(rep=rep, model="HistGradientBoosting", seed=seed,
                            **score(hgb.predict_proba(X_va)[:, 1])))
        print(f"[{rep}] HistGradientBoosting seed={seed} done")

res = pd.DataFrame(results)

{'A': 43, 'B': 77} features
[A] LogisticRegression done
[A] HistGradientBoosting seed=42 done
[A] HistGradientBoosting seed=43 done
[A] HistGradientBoosting seed=44 done
[B] LogisticRegression done
[B] HistGradientBoosting seed=42 done
[B] HistGradientBoosting seed=43 done
[B] HistGradientBoosting seed=44 done


## 5. A vs B
If the B - A gain is no bigger than the seed-to-seed std, it is not distinguishable from noise.
Do not claim an improvement in the report unless it clearly exceeds that.

In [9]:
def mean_std(g, col):
    m, s = g[col].mean(), g[col].std()
    return f"{m:.4f}" + (f" +/- {s:.4f}" if len(g) > 1 else "")

print(f"{'Model':<22}{'Rep':<5}{'PR-AUC':<20}{'F1 @0.5':<20}{'runs'}")
for (model, rep), g in res.groupby(["model", "rep"]):
    print(f"{model:<22}{rep:<5}{mean_std(g, 'pr_auc'):<20}{mean_std(g, 'f1'):<20}{len(g)}")

print("\nB minus A (PR-AUC):")
for model, g in res.groupby("model"):
    a = g[g.rep == "A"]["pr_auc"].mean()
    b = g[g.rep == "B"]["pr_auc"].mean()
    noise = g.groupby("rep")["pr_auc"].std().max()
    noise_txt = f"seed std ~ {noise:.4f}" if pd.notna(noise) else "single run, no noise estimate"
    print(f"  {model:<22}{b - a:+.4f}  ({100 * (b - a) / a:+.1f}% relative; {noise_txt})")

Model                 Rep  PR-AUC              F1 @0.5             runs
HistGradientBoosting  A    0.0869 +/- 0.0033   0.1124 +/- 0.0011   3
HistGradientBoosting  B    0.0900 +/- 0.0016   0.1150 +/- 0.0007   3
LogisticRegression    A    0.0686              0.0775              1
LogisticRegression    B    0.0750              0.0862              1

B minus A (PR-AUC):
  HistGradientBoosting  +0.0031  (+3.6% relative; seed std ~ 0.0033)
  LogisticRegression    +0.0064  (+9.4% relative; single run, no noise estimate)
